# PCA via SVD from scratch

## 1. First-principles

Given $x \in \mathbb{R}^{n \times d}$

**Principal Component Analysis (PCA)**

- a principal component is a direction in space that represent a perspective on an object.
- PCA is proces constructs new orthogonal axes (aka principal components).
- To derive principal components:
  - First principal component: direction along which the data varies the most
  - Second principal component: direction of greatest remaining varied data, which perpendicular to the first.
  - ...
  - Until we have:

$$
\min(n-1,d)
$$

non-zero principal components.

Old coordiates:

$$
x = \begin{bmatrix}
    x_1 \\ x_2 \\ \vdots \\ x_d
\end{bmatrix}
$$

to new coordinates:

$$
z = \begin{bmatrix}
    z_1 \\ z_2 \\ \vdots \\x_k
\end{bmatrix}
$$

with $v_j$ := j-th principal direction:

$$
z_j = x^T v_j
$$

## 2. Variance in PCA

- PCA analyses **correlation between components**:
  - High variance -> meaningful component -> keep
  - Low variance -> points' variance around 0 -> don't keep

Start with the centering -> **Centred observation**:

$$
\mu = \frac{1}{n}\sum_j x_j \Rightarrow (X_c)_{ij} = X_{ij} - \mu
$$

Then after centering:

$$
\frac{1}{n}\sum_{i=1}^n(x_i - \mu) = 0
$$

For unit direction $v \in \mathbb{R}^d, \|v\|_2 = 1$, projection of $x_i$ onto $v$ is:

$$
z_i = x_i^Tv
$$

for complete dataset:

$$
z = X_c v
$$

with:

$$
X_c \in \mathbb{R}^{n \times d}, v \in \mathbb{R}^d, z \in \mathbb{R}^n
$$

- $X_c$ is centred -> $z$ is centred -> $\overline{z} = 0$.

$z$ is **sample**, so Variance of z is:

$$
\text{Var}(z) = \frac{1}{n - 1} \sum(z_i - \overline{z})^2 = \frac{1}{n-1}\sum z_i^2 = \frac{1}{n-1}z^T z
$$

$$
\text{Var}(z) = \frac{1}{n-1} (X_c v)^T(X_c v)
$$

Covariance matrix $C = \frac{1}{n - 1} X_c^T X_c$:

$$
\text{Var}(z) = v^T C v
$$

=> PCA finds max $v^T Cv$.


### How to solve the $\max(v^T Cv)$ with Eigenvectors

#### Lagrange Multipliers with Constraint Optimisations

Suppose $f(x, y) = x^2 + y^2$. The minimum $f$ is when

$$
\nabla f(x, y) = 2x + 2y = 0
$$

or $x^* = y^* = 0$

Now suppose constraint:

$$
h(x, y) = x + y - 1 = 0
$$

Then $(x^*, y^*) = (0, 0)$ is no longer the optimum point. We have to replace $y = 1 - x$ to solve the actual min:

$$
(x^*, y^*) = (\frac{1}{2}, \frac{1}{2}) \Rightarrow \nabla f(x^*, y^*) = \begin{bmatrix}
    2x^* \\
    2y^*
\end{bmatrix} = \begin{bmatrix}
    1 \\ 1
\end{bmatrix}
$$

> With constraint surface $h(x)$, $\nabla f$ is no longer use to  identify the optimum point.

**Imagine:** walking on a circular path on a mountain. If at some point, $\nabla f$ point upward to the current path, you only go forward and backward -> never reach the peak of mountain. -> Local optimum.

> So $\nabla f$ point the same direction with $\nabla h$, which is perpendicular to the constraint surface.

**Lagrange Multiplier**

$$
\nabla f(x^*) + \lambda\nabla h(x^*) = 0
$$

with:
- $\lambda$: Lagrange multiplier

#### Lagrangian

For m constraints: $h_1(x),...,h_m(x)$, the Lagrangian:

$$
\mathcal{L}(x, \lambda) = f(x) + \sum_{i=1}^m \lambda_i h_i(x) = f(x) + \lambda^T h(x)
$$

for:

$$
h(x) = \begin{bmatrix}
    h_1(x) \\
    \vdots \\
    h_m(x)
\end{bmatrix}
$$

or stationary condition

$$
\nabla f(x^*) + \sum_{i=1}^m \lambda_i^* \nabla h_i(x^*) = 0
$$

Hence:

$$
\nabla f(x^*) \in \text{span} \{ \nabla h_1(x^*), ..., \nabla h_m(x^*) \}
$$

#### Now go back to the original equation

We have

$$
f(v) = v^T Cv = 
$$

Since $v^Tv = 1$, suppose constraint function:

$$
h(v) = v^Tv - 1 = 0
$$

Apply Lagrange multiplier

$$
\mathcal{L}(v, \lambda) = v^TCv - \lambda(v^Tv - 1)
$$

Differentiate $\mathcal{L}$ over $v$:

$$
\nabla_v \mathcal{L} = 2Cv - 2\lambda v
$$

Set $nabla_v \mathcal{L} = 0$, we have Eigenvalue equation:

$$
Cv = \lambda v
$$

> Therefore:
> - principal directions are **eigenvectors** of covariance matrix;
> - variance along each direction is **eigenvalue**.

### How SVD solve PCA?

For $X_c \in \mathbb{R}^{n\times d}$:

$$
X_c = U\Sigma V^T
$$

with

$$
U \in \mathbb{R}^{n \times r}, \qquad \Sigma \in \mathbb{R}^{r \times r}, \qquad V^T \in \mathbb{R}^{r\times d}
$$

where:

$$
r = \text{rank}(X_c) \le \min(n, d)
$$

- $r$ columns: directions in feature space
- diagonal values of $\Sigma$: singular values

In numpy

```python
import numpy as np

U, singular_values, Vt = np.linalg.svd(X_centred, full_matrices=False)
```

Now for Covariance matrix:

$$
C = \frac{1}{n-1}X_c^T X_c = \frac{1}{n-1}(U\Sigma V^T)^T(U\Sigma V^T) = V\Sigma^TU^TU\Sigma V^T
$$

But $U^TU = 1$, hence:

$$
C = V\frac{\Sigma^2}{n-1}V^T
$$

Which means:

$$
\boxed{
    \text{principal directions } = \text{ columns of } V
}
$$

and:

$$
\lambda_j = \frac{\sigma_j^2}{n-1}
$$

with:
- $\sigma$: j-th singular value
- $\lambda$: variance explained by the j-th component